In [1]:
%load_ext autoreload
%autoreload 2

# general
from pathlib import Path
import re
import numpy as np
import datetime
import matplotlib.pyplot as plt

# spatial
import xarray as xa

# custom
import cbsyst as cb
from cmipper import functions_creche, utils, config, parallelised_download_and_process, main, file_ops

/maps/rt582/miniforge3/envs/shiftpy/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# Integrating `esgpull`

`esgpull` provides streamlined, maintained functionality to search and download files from the `esgf-node` servers. The job of this notebook is to substitute my own downloading schema (developed as `cmipper`) for `esgpull` while retaining the additional functionality I require. This is:
- ~~Consistent, human-readable file labelling~~ -> Happy with what's given, keeping for simplicity
- Regridding from variable grid to fixed grid
- POTENTIALLY – concatenating and cropping variables

This should enable faster, more reliable, and better-logged downloading of all necessary GCM variables in a 'one file' approach. The user should be able to specify which variables (at what resolution, over a specified time period), which model ensembles, and how many model runs they want for each. The program should then attempt to download these N times.

### Variables
| variable_long_name                                           | variable_id | variable_units | frequency                   |
|--------------------------------------------------------------|-------------|----------------|-----------------------------|
|                                                              |             |                |                             |
| Sea Surface Temperature                                      | tos         | [degC]         | 3hr, Amon, Oday, Odec, **Omon** |
| Sea Water Potential Temperature                              | thetao      | [degC]         | Omon                        |
| Downwelling Shortwave Radiation in Sea Water                 | rsdo        | [W m-2]        | Omon                        |
| Sea Water Salinity                                           | so          | [0.001]        | Omon                        |
| Sea Water X Velocity                                         | uo          | [m s-1]        | Odec, **Omon**                  |
| Sea Water Y Velocity                                         | vo          | [m s-1]        | Odec, **Omon**                  |
| Aragonite Concentration                                      | arag        | [mol m-3]      | Oyr                         |
| Dissolved Nitrate Concentration                              | no3         | [mol m-3]      | **Omon**, Oyr                   |
| Total Dissolved Inorganic Phosphorus Concentration           | po4         | [mol m-3]      | **Omon**, Oyr                   |

Working list:
1. ~~For an example ensemble (`HadGEM3-GC31-MM`) and variable (`tos`), search and download via command-line~~
2. ~~Automate this with a bash script~~ Done to some extent by generating query from yaml file
3. Pipe downloads into regridder

Example query:
`$ esgpull search project:CMIP6 experiment_id:historical institution_id:HadGEM3-GC31-MM variable_id:tos table_id:Omon member_id:r1i1p1f1 --distrib true --show`

In [142]:
# creating searchs from yaml

import itertools
import yaml
import subprocess

# Load YAML config file
with open('/maps/rt582/cmipper/tmp/data_to_download.yaml', 'r') as file:
    config = yaml.safe_load(file)

# Start the command as a list
query = ["esgpull", "search"]

# Add parameters, handling spaces by keeping each argument separate
for param, value in config.items():
    if not isinstance(value, list):
        # If there's a space, wrap value in quotes
        query.append(f"{param}:{value}" if ' ' not in str(value) else f"{param}:'{value}'")
    else:
        # Join list values with commas and wrap if they have spaces
        joined_values = ','.join([f"'{v}'" if ' ' in str(v) else v for v in value])
        query.append(f"{param}:{joined_values}")

# Add the distribution flag
query.append("--distrib")
query.append("true")

# Display the final command
print("Running:\n", ' '.join(query))

# Uncomment to execute the command
# subprocess.run(query)

Running:
 esgpull search project:CMIP6 nominal_resolution:'25 km' realm:atmos,ocean source_id:ECMWF-IFS-HR institution_id:ECMWF experiment_id:hist-1950 table_id:Omon,Amon variable_id:tos,rsds,so,uo --distrib true


# Testing processing functions

In [111]:
# write script/function to process (regrid) files at it becomes available

# open test
import xarray as xa
from cmipper import processing

test_fp = "/maps/rt582/cmipper/.esgpull/data/CMIP6/HighResMIP/ECMWF/ECMWF-IFS-HR/hist-1950/r1i1p1f1/Omon/vo/gn/v20170915/vo_Omon_ECMWF-IFS-HR_hist-1950_r1i1p1f1_gn_195001-195012.nc"
# select level
out = processing.extract_dataset_level(test_fp, 0)

# Processing resulting files

In [127]:
raw_data_dir = Path('/maps/rt582/cmipper/.esgpull/data/')
processed_data_dir = Path('/maps/rt582/cmipper/data/test/')

file_path = Path(fp)
rel_path = file_path.relative_to(raw_data_dir)

output_path = processed_data_dir / rel_path
output_path.parent.mkdir(parents=True, exist_ok=True)

# processing metadata
with open('/maps/rt582/cmipper/tmp/data_processing.yaml', 'r') as file:
    data_processing_config = yaml.safe_load(file)

select_level = data_processing_config["select_level"]
do_regrid = data_processing_config["do_regrid"]
output_grid = data_processing_config["output_grid"]
remap_method = data_processing_config["remap_method"]

In [ ]:
remap_template_fp = output_path.parent / "remap_template.txt"
# generate remap file if necessary
remap_template_fp = utils.return_remap_template(input_file=out, remap_template_fp=remap_template_fp, out_grid=output_grid)

In [137]:
test = processing.reproject_xa_d(xa_d=out, ds_fp=test_fp, output_grid=output_grid, remap_method=remap_method)

Using existing remapping template at  /maps/rt582/cmipper/.esgpull/data/CMIP6/HighResMIP/ECMWF/ECMWF-IFS-HR/hist-1950/r1i1p1f1/Omon/vo/gn/v20170915/latlon_remap_template.txt
